# Google: 'LK Hadith Corpus Github'

In [2]:
!git clone https://github.com/ShathaTm/LK-Hadith-Corpus.git

'git' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import glob
path = '/content/LK-Hadith-Corpus'
files = sorted(glob.glob(path+'//**//*.csv', recursive=True))

In [3]:
import re

def clean_text(text):
  text = text.lower()
  text = re.sub(r'[^a-zA-Z0-9\s]', '', text) # remove punctuations
  text = re.sub(r'\s+', ' ', text)           # removes extra space
  return text

In [4]:
columns = [
    'Chapter_Number', 'Chapter_English', 'Chapter_Arabic',
    'Section_Number', 'Section_English', 'Section_Arabic',
    'Hadith_Number',
    'English_Hadith', 'English_Isnad', 'English_Matn', 'English_Grade',
    'Arabic_Hadith', 'Arabic_Isnad', 'Arabic_Matn', 'Arabic_Grade'
]

In [5]:
import pandas as pd
columns.append('Cleaned_Hadith')
all_hadith = []

for file in files:
  df = pd.read_csv(file, names=columns, skiprows=1)
  df['Cleaned_Hadith'] = df['English_Hadith'].astype(str).apply(clean_text)

  all_hadith.extend(df[columns].values.tolist())
  # all_hadith.append(df[['Chapter_Number', 'Chapter_English']])

In [6]:
hadith_df = pd.DataFrame(all_hadith, columns=columns)
hadith_df.head(2)

,Chapter_Number,Chapter_English,Chapter_Arabic,Section_Number,Section_English,Section_Arabic,Hadith_Number,English_Hadith,English_Isnad,English_Matn,English_Grade,Arabic_Hadith,Arabic_Isnad,Arabic_Matn,Arabic_Grade,Cleaned_Hadith
0,1.0,Purification (Kitab Al-Taharah),كتاب الطهارة,1.0,Seclusion While Relieving Oneself,باب التَّخَلِّي عِنْدَ قَضَاءِ الْحَاجَةِ,1.0,Narrated Mughirah ibn Shu'bah: When the Prophe...,Narrated Mughirah ibn Shu'bah:,When the Prophet (ﷺ) went (outside) to relieve...,حَدَّثَنَا عَبْدُ اللَّهِ بْنُ مَسْلَمَةَ بْنِ...,حَدَّثَنَا عَبْدُ اللَّهِ بْنُ مَسْلَمَةَ بْنِ...,أَنَّ النَّبِيَّ صلى الله عليه وسلم كَانَ إِذَ...,NaN,Hasan Sahih,narrated mughirah ibn shubah when the prophet ...
1,1.0,Purification (Kitab Al-Taharah),كتاب الطهارة,1.0,Seclusion While Relieving Oneself,باب التَّخَلِّي عِنْدَ قَضَاءِ الْحَاجَةِ,2.0,Narrated Jabir ibn Abdullah: When the Prophet ...,Narrated Jabir ibn Abdullah:,When the Prophet (ﷺ) felt the need of relievin...,حَدَّثَنَا مُسَدَّدُ بْنُ مُسَرْهَدٍ، حَدَّثَن...,حَدَّثَنَا مُسَدَّدُ بْنُ مُسَرْهَدٍ، حَدَّثَن...,أَنَّ النَّبِيَّ صلى الله عليه وسلم كَانَ إِذَ...,NaN,Sahih - Authentic,narrated jabir ibn abdullah when the prophet f...


In [7]:
hadith_df.to_csv('cleaned_hadith_data.csv', index=False)

In [8]:
!pip install sentence-transformers

In [9]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
hadith_df.columns

Index(['Chapter_Number', 'Chapter_English', 'Chapter_Arabic', 'Section_Number',
       'Section_English', 'Section_Arabic', 'Hadith_Number', 'English_Hadith',
       'English_Isnad', 'English_Matn', 'English_Grade', 'Arabic_Hadith',
       'Arabic_Isnad', 'Arabic_Matn', 'Arabic_Grade', 'Cleaned_Hadith'],
      dtype='object')

In [11]:
embeddings = model.encode(hadith_df['Cleaned_Hadith'].values)

In [12]:
import numpy as np
embeddings = np.array(embeddings)

In [13]:
np.save('hadith_embeddings.npy', embeddings)

In [14]:
embeddings = np.load('hadith_embeddings.npy')

In [15]:
# FAISS - Facebook AI for Similarity Search
# !pip install faiss-gpu
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.4 MB/s eta 0:00:00


In [16]:
import faiss

dimensions = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimensions)   # Euclidean Distance

In [17]:
faiss_index.add(embeddings)

In [18]:
faiss.write_index(faiss_index, "faiss_index.index")

In [19]:
def get_similar_hadith(query, count=5, model=model, faiss_index=faiss_index):
  query_embedding = model.encode([query])
  distance, indices = faiss_index.search(query_embedding, count)

  for i in range(count):
    print(f"Hadith {i+1}, Distance: {distance[0][i]}")
    print(hadith_df['English_Hadith'].iloc[indices[0][i]])

In [20]:
get_similar_hadith("How many prayers are there?")

Hadith 1, Distance: 0.7171144485473633
It was narrated that Ibn ‘Abbas said: “Your Prophet (ﷺ) was enjoined to do fifty prayers but he returned to your Lord to make (i.e., reduce) them to five prayers.”
Hadith 2, Distance: 0.7440201640129089
Abu Huraira reported Allah's Messenger (ﷺ) as saying: Prayer said in a congregation is equivalent to twenty-five (prayers) as compared with the prayer said by a single person.
Hadith 3, Distance: 0.7486099004745483
Narrated `Abdullah bin `Umar: Allah's Messenger (ﷺ) said, "The prayer in congregation is twenty seven times superior to the prayer offeredby person alone."
Hadith 4, Distance: 0.8015742301940918
It was narrated from Abu Hurairah that: The Messenger of Allah said: "The prayer in congregation is twenty-five times more virtuous than the prayer of anyone of you on his own."
Hadith 5, Distance: 0.8022949695587158
Narrated Abdullah ibn Umar: There were fifty prayers (obligatory in the beginning); and (in the beginning of Islam) washing seven t